# 03 - Casos de cierre desde splits reales

Este notebook conecta `realtime/` con datos livianos del proyecto: usa solo los CSV de splits para generar casos de evaluacion de cierre.

No abre videos, no carga ROIs `.npz`, no toca checkpoints y no modifica datos de entrenamiento.

## 1. Setup y split disponible

El split se lee como CSV. La columna `npz` queda como referencia, pero no se carga ningun archivo pesado.

In [1]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "realtime" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value) or "-"
    text = str(value)
    return text.replace("\n", " ").replace("|", "\\|")


def show_table(rows, columns):
    if not rows:
        display(Markdown("_Sin filas para mostrar._"))
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_fmt(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))


def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

print(f"Repo detectado: {ROOT.name}")
from realtime.src.cierre import HeuristicClosureProvider
from realtime.src.dataset_cierre import build_cases_from_split_csv, write_jsonl
from realtime.src.evaluar_cierre import evaluate_closure, load_cases

split_path = ROOT / "vsr_models" / "splits" / "val.csv"
out_dir = ROOT / "realtime" / "outputs" / "notebooks" / "03_splits"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "cierre_val_60.jsonl"

print("Split:", split_path.relative_to(ROOT))
print("Existe split:", split_path.exists())
print("Output JSONL:", out_path.relative_to(ROOT))

Repo detectado: labios-argentos
Split: vsr_models\splits\val.csv
Existe split: True
Output JSONL: realtime\outputs\notebooks\03_splits\cierre_val_60.jsonl


## 2. Muestra de filas reales

Esto permite ver de donde salen los textos sin materializar datos pesados.

In [2]:
with split_path.open("r", encoding="utf-8", newline="") as fh:
    reader = csv.DictReader(fh)
    split_rows = list(reader)

sample_rows = []
for row in split_rows[:5]:
    sample_rows.append({
        "split": row.get("split"),
        "spk": row.get("spk"),
        "clip": row.get("clip"),
        "n_frames": row.get("n_frames"),
        "texto": row.get("texto"),
        "npz_referenciado": row.get("npz", "")[:70] + "...",
    })

print("Filas en val.csv:", len(split_rows))
show_table(sample_rows, ["split", "spk", "clip", "n_frames", "texto", "npz_referenciado"])

Filas en val.csv: 466


| split | spk | clip | n_frames | texto | npz_referenciado |
| --- | --- | --- | --- | --- | --- |
| val | f02 | clip_0000 | 128 | es una verguenza absoluta lo que acaba de pasar en la cancha de racing | data/processed/lip_rois/AZZARO REACCIÓN - CICLO TERMINADO RACING EMPAT... |
| val | f02 | clip_0001 | 139 | y la verdad es que cualquier cosa que pueda decir en este video | data/processed/lip_rois/AZZARO REACCIÓN - CICLO TERMINADO RACING EMPAT... |
| val | f02 | clip_0002 | 157 | no no me voy a arrepentir no no me voy a arrepentir bajo ningun concepto | data/processed/lip_rois/AZZARO REACCIÓN - CICLO TERMINADO RACING EMPAT... |
| val | f02 | clip_0003 | 264 | porque tengo los huevos llenos de este equipo de racing y la realidad es que no hay manera de que el tecnico pueda seguir despues de quedar afuera | data/processed/lip_rois/AZZARO REACCIÓN - CICLO TERMINADO RACING EMPAT... |
| val | f02 | clip_0006 | 105 | ahora no sali en ni segundo es una cosa inaudita | data/processed/lip_rois/AZZARO REACCIÓN - CICLO TERMINADO RACING EMPAT... |

## 3. Generacion de casos de cierre

Por cada texto se crean prefijos incompletos (`wait`), conectores colgantes (`wait`) y frases completas suficientemente largas (`commit`).

In [3]:
cases = build_cases_from_split_csv(split_path, limit=60)
write_jsonl(cases, out_path)
loaded_cases = load_cases(out_path)

case_counts = Counter(case["case"] for case in loaded_cases)
action_counts = Counter(case["expected_action"] for case in loaded_cases)
show_table(
    [{"tipo": key, "cantidad": value} for key, value in sorted(case_counts.items())],
    ["tipo", "cantidad"],
)
show_table(
    [{"accion_esperada": key, "cantidad": value} for key, value in sorted(action_counts.items())],
    ["accion_esperada", "cantidad"],
)
print("Casos generados:", len(loaded_cases))

| tipo | cantidad |
| --- | --- |
| dangling_wait | 18 |
| full_commit | 20 |
| prefix_wait | 22 |

| accion_esperada | cantidad |
| --- | --- |
| commit | 20 |
| wait | 40 |

Casos generados: 60


## 4. Muestra de casos generados

La muestra combina prefijos y textos completos, manteniendo trazabilidad al split original.

In [4]:
sample_cases = []
for case in loaded_cases[:10]:
    sample_cases.append({
        "case": case["case"],
        "expected": case["expected_action"],
        "partial_text": case["partial_text"],
        "segment_id": case["segment_id"],
    })
show_table(sample_cases, ["case", "expected", "partial_text", "segment_id"])

| case | expected | partial_text | segment_id |
| --- | --- | --- | --- |
| prefix_wait | wait | es una verguenza absoluta | val_0001_clip_0000_prefix |
| dangling_wait | wait | es una | val_0001_clip_0000_dangling_2 |
| full_commit | commit | es una verguenza absoluta lo que acaba de pasar en la cancha de racing | val_0001_clip_0000_full |
| prefix_wait | wait | y la verdad es | val_0002_clip_0001_prefix |
| dangling_wait | wait | y la | val_0002_clip_0001_dangling_2 |
| full_commit | commit | y la verdad es que cualquier cosa que pueda decir en este video | val_0002_clip_0001_full |
| prefix_wait | wait | no no me voy | val_0003_clip_0002_prefix |
| dangling_wait | wait | no no me voy a | val_0003_clip_0002_dangling_5 |
| full_commit | commit | no no me voy a arrepentir no no me voy a arrepentir bajo ningun concepto | val_0003_clip_0002_full |
| prefix_wait | wait | porque tengo los huevos | val_0004_clip_0003_prefix |

## 5. Evaluacion de la baseline sobre casos del split

Esto no reemplaza una evaluacion con salidas reales del VSR, pero deja un benchmark inicial reproducible con textos del proyecto.

In [5]:
summary = evaluate_closure(loaded_cases, HeuristicClosureProvider())
metric_rows = [
    {"metrica": "casos", "valor": summary["count"]},
    {"metrica": "accuracy", "valor": summary["accuracy"]},
    {"metrica": "precision_commit", "valor": summary["commit_precision"]},
    {"metrica": "recall_commit", "valor": summary["commit_recall"]},
    {"metrica": "commits_prematuros", "valor": summary["premature_commit_rate"]},
    {"metrica": "waits_innecesarios", "valor": summary["unnecessary_wait_rate"]},
    {"metrica": "latencia_p50_ms", "valor": summary["latency_ms"]["p50"]},
    {"metrica": "latencia_p95_ms", "valor": summary["latency_ms"]["p95"]},
    {"metrica": "fallbacks", "valor": summary["fallbacks"]},
]
show_table(metric_rows, ["metrica", "valor"])

| metrica | valor |
| --- | --- |
| casos | 60 |
| accuracy | 1.0000 |
| precision_commit | 1.0000 |
| recall_commit | 1.0000 |
| commits_prematuros | 0.0000 |
| waits_innecesarios | 0.0000 |
| latencia_p50_ms | 0.0417 |
| latencia_p95_ms | 0.1259 |
| fallbacks | 0 |

## 6. Errores o casos a revisar

Si aparecen errores, esta tabla muestra donde mirar primero. Si no aparecen, igual conviene ampliar el benchmark con predicciones reales del VSR.

In [6]:
errors = [row for row in summary["rows"] if not row["correct"]]
error_rows = []
for row in errors[:10]:
    error_rows.append({
        "segment_id": row["segment_id"],
        "case": row["case"],
        "expected": row["expected"],
        "predicted": row["predicted"],
        "reason": row["reason"],
        "risk_flags": row["risk_flags"],
    })
show_table(error_rows, ["segment_id", "case", "expected", "predicted", "reason", "risk_flags"])
print("Errores encontrados:", len(errors))

_Sin filas para mostrar._

Errores encontrados: 0


## 7. Verificacion de alcance

La prueba usa solo `vsr_models/splits/val.csv` y escribe un JSONL ignorado por Git. Las rutas `.npz` quedan referenciadas como metadata, pero no se abren.

In [7]:
check = {
    "split_leido": str(split_path.relative_to(ROOT)),
    "jsonl_generado": str(out_path.relative_to(ROOT)),
    "jsonl_existe": out_path.exists(),
    "npz_cargados": 0,
    "videos_cargados": 0,
    "checkpoints_tocados": 0,
}
show_json(check)

{
  "checkpoints_tocados": 0,
  "jsonl_existe": true,
  "jsonl_generado": "realtime\\outputs\\notebooks\\03_splits\\cierre_val_60.jsonl",
  "npz_cargados": 0,
  "split_leido": "vsr_models\\splits\\val.csv",
  "videos_cargados": 0
}


## 8. Lectura final

Resultado de este notebook:

- Ya hay un puente reproducible entre splits reales y evaluacion de cierre.
- El benchmark inicial corre sin datos pesados.
- El siguiente paso serio es reemplazar estos casos derivados por salidas reales del VSR + ground truth cuando Joaquin/Mateo tengan artefactos coordinados.
- Mientras tanto, esta baseline queda lista para comparar contra un provider LLM sin cambiar el contrato.